In [121]:
import numpy as np
import pandas as pd
import requests
import time
import json
import os
import re
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import kagglehub
from kagglehub import KaggleDatasetAdapter
import warnings

warnings.filterwarnings("ignore")

TOP_N = 10
MIN_RATINGS = 50 # ignore books rated by fewer than this many users
CACHE_PATH = "../data/isbn_cache.json"
ISBNDB_API_KEY = "65936_e138f82fe91ef26ef1821e3299dd945b"

USE_CSV = False   
RATINGS_CSV = "../data/books_rated.csv"

Load Kaggle Data

In [122]:
print("loading Books.csv")
books = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "arashnic/book-recommendation-dataset",
    "Books.csv",
    pandas_kwargs={"on_bad_lines": "skip", "encoding": "latin-1"},
)

print("loading Ratings.csv")
ratings = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "arashnic/book-recommendation-dataset",
    "Ratings.csv",
    pandas_kwargs={"encoding": "latin-1"},
)

# drop rating == 0
ratings = ratings[ratings["Book-Rating"] > 0].copy()

print(f"books: {len(books):,}")
print(f"ratings: {len(ratings):,}")
print(f"users: {ratings['User-ID'].nunique():,}")

loading Books.csv
loading Ratings.csv
books: 271,360
ratings: 433,671
users: 77,805


Clean and Filter

In [123]:
books = books.rename(columns={
    "Book-Title": "title",
    "Book-Author": "author",
    "ISBN": "isbn",
})

def clean(s):
    return str(s).strip() if pd.notna(s) else ""

books["title"] = books["title"].apply(clean)
books["author"] = books["author"].apply(clean)

# filter to popular books
rating_counts = ratings.groupby("ISBN")["Book-Rating"].count().rename("n_ratings")
popular_isbns = rating_counts[rating_counts >= MIN_RATINGS].index

books = books[books["isbn"].isin(popular_isbns)].drop_duplicates("isbn").reset_index(drop=True)
print(f"after popularity filter: {len(books):,}")

# strip parentheticals and subtitles, then keep the edition with the most ratings
def normalize_title(t):
    t = t.lower().strip()
    t = re.sub(r'\s*\([^)]*\)', '', t)
    t = re.sub(r'\s*:.*$', '', t)
    return re.sub(r'\s+', ' ', t).strip()

books["_title_norm"] = books["title"].apply(normalize_title)
books["_n_ratings"]  = books["isbn"].map(rating_counts).fillna(0).astype(int)

books = (
    books.sort_values("_n_ratings", ascending=False)
         .drop_duplicates(subset="_title_norm")
         .drop(columns=["_title_norm", "_n_ratings"])
         .reset_index(drop=True)
)
print(f"after deduplication: {len(books):,}")
books[["isbn", "title", "author"]].head(5)

after popularity filter: 531
after deduplication: 462


,isbn,title,author
0,0316666343,The Lovely Bones: A Novel,Alice Sebold
1,0971880107,Wild Animus,Rich Shapero
2,0385504209,The Da Vinci Code,Dan Brown
3,0312195516,The Red Tent (Bestselling Backlist),Anita Diamant
4,0060928336,Divine Secrets of the Ya-Ya Sisterhood: A Novel,Rebecca Wells


Enrich Metadata via ISBNdb

In [124]:
RATE_LIMIT = 1.0 # to avoid rate limits
def load_cache():
    if os.path.exists(CACHE_PATH):
        with open(CACHE_PATH) as f:
            return json.load(f)
    return {}

def save_cache(cache):
    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    with open(CACHE_PATH, "w") as f:
        json.dump(cache, f, indent=2)


def fetch_isbndb(isbn):
    # Fetch a single book from ISBNdb
    r = requests.get(
        f"https://api2.isbndb.com/book/{isbn}",
        headers={"Authorization": ISBNDB_API_KEY},
        timeout=10,
    )
    if r.status_code == 404:
        return None
    if r.status_code == 429:
        print("\nrate limited")
        time.sleep(5)
        return fetch_isbndb(isbn)   # retry once
    r.raise_for_status()
    book = r.json().get("book", {})
    return {
        "synopsis": (book.get("synopsis") or "").strip(),
        "subjects": book.get("subjects") or [],
    }


cache = load_cache()
to_fetch = [isbn for isbn in books["isbn"] if isbn not in cache]

print(f"total books: {len(books):,}")
print(f"cached: {len(cache):,}")
print(f"to fetch: {len(to_fetch):,}\n")

for isbn in tqdm(to_fetch, desc="ISBNdb"):
    try:
        result = fetch_isbndb(isbn)
        cache[isbn] = result if result else {"synopsis": "", "subjects": []}
    except Exception as exc:
        print(f"\n  [warn] {isbn} failed: {exc}")
        cache[isbn] = {"synopsis": "", "subjects": []}
    time.sleep(RATE_LIMIT)

save_cache(cache)
print("cache saved")

books["synopsis"] = books["isbn"].map(lambda x: cache.get(x, {}).get("synopsis", ""))
books["subjects"] = books["isbn"].map(
    lambda x: ", ".join(cache.get(x, {}).get("subjects", [])[:8])
)

# rebuild text: title + author always present; synopsis + subjects when available
def build_text(row):
    parts = [f"{row['title']} by {row['author']}" if row["author"] else row["title"]]
    if row["synopsis"]:
        parts.append(row["synopsis"])
    if row["subjects"]:
        parts.append(f"Genres: {row['subjects']}")
    return " | ".join(parts)

books["text"] = books.apply(build_text, axis=1)

syn_cov  = (books["synopsis"] != "").sum()
subj_cov = (books["subjects"] != "").sum()
print(f"\ncoverage:")
print(f"synopsis : {syn_cov:,} / {len(books):,}  ({syn_cov / len(books) * 100:.1f}%)")
print(f"subjects : {subj_cov:,} / {len(books):,}  ({subj_cov / len(books) * 100:.1f}%)")

total books: 462
cached: 531
to fetch: 0



ISBNdb: 0it [00:00, ?it/s]

cache saved

coverage:
synopsis : 460 / 462  (99.6%)
subjects : 460 / 462  (99.6%)


Embed with Sentence-BERT

In [125]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"embedding {len(books):,} books...")
embeddings = model.encode(
    books["text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding 462 books...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

shape: (462, 384)


My Preferences

In [126]:
def search_isbndb_by_title(title):
    r = requests.get(
        f"https://api2.isbndb.com/books/{requests.utils.quote(title)}",
        headers={"Authorization": ISBNDB_API_KEY},
        params={"pageSize": 5},
        timeout=10,
    )
    if r.status_code != 200:
        return None
    results = r.json().get("books", [])
    if not results:
        return None
    title_lower = title.lower()
    best = next(
        (b for b in results if b.get("title", "").lower() == title_lower),
        results[0]
    )
    return {
        "title": best.get("title", title),
        "authors": ", ".join(best.get("authors") or []),
        "synopsis": (best.get("synopsis") or "").strip(),
        "subjects": best.get("subjects") or [],
    }

CSV_META = {}

if USE_CSV:
    csv_df = pd.read_csv(RATINGS_CSV)

    # auto-detect columns
    csv_title_col = next((c for c in csv_df.columns if "title"    in c.lower()), None)
    csv_rating_col = next((c for c in csv_df.columns if "rating"   in c.lower()), None)
    csv_authors_col = next((c for c in csv_df.columns if "author"   in c.lower()), None)
    csv_synopsis_col = next((c for c in csv_df.columns if "synopsis" in c.lower()), None)
    csv_subjects_col = next((c for c in csv_df.columns if "subject"  in c.lower()), None)

    if not csv_title_col or not csv_rating_col:
        raise ValueError(f"could not find title/rating columns in {RATINGS_CSV}. columns: {list(csv_df.columns)}")

    csv_df = csv_df[csv_df[csv_rating_col].notna()].copy()

    MY_RATINGS = dict(zip(
        csv_df[csv_title_col].str.strip(),
        csv_df[csv_rating_col].astype(float).round().astype(int).clip(1, 5)
    ))

    # store per-book metadata so the loop below can use it
    for idx, row in csv_df.iterrows():
        t = str(row[csv_title_col]).strip()
        CSV_META[t] = {
            "authors":  str(row[csv_authors_col]).strip()  if csv_authors_col  and pd.notna(row[csv_authors_col])  else "",
            "synopsis": str(row[csv_synopsis_col]).strip() if csv_synopsis_col and pd.notna(row[csv_synopsis_col]) else "",
            "subjects": str(row[csv_subjects_col]).strip() if csv_subjects_col and pd.notna(row[csv_subjects_col]) else "",
        }

    print(f"loaded {len(MY_RATINGS)} rated books from {RATINGS_CSV}\n")

else:
    MY_RATINGS = {
        "Harry Potter and the Sorcerer's Stone": 5,
        "The Hobbit": 5,
        "Divergent": 5,
        "Bridget Jones's Diary": 4,
        "The Da Vinci Code": 3,
        "Twilight": 1,
        "To Kill a Mockingbird": 2,
        "The Great Gatsby": 2,
        "Pride and Prejudice": 1,
    }

RATE_LIMIT = 1.0  # seconds between ISBNdb requests

title_lower_to_idx = {row["title"].lower(): i for i, row in books.iterrows()}

profile_vecs = []  
profile_weights = []  
rated_titles = []
rated_raw_ratings = []   
dataset_rated_indices = []  

for title, rating in MY_RATINGS.items():
    w = float(rating) if rating >= 4 else (1.0 if rating == 3 else -0.5)

    # find in Kaggle dataset
    idx = title_lower_to_idx.get(title.lower())
    if idx is None:
        matches = [i for t, i in title_lower_to_idx.items() if title.lower() in t]
        if matches:
            idx = matches[0]

    if idx is not None:
        profile_vecs.append(embeddings[idx])
        profile_weights.append(w)
        rated_titles.append(books.loc[idx, "title"])
        rated_raw_ratings.append(rating)
        dataset_rated_indices.append(idx)
        print(f"  [{rating} Stars] {books.loc[idx, 'title']} by {books.loc[idx, 'author']}")
        continue

    # not in dataset, check CSV for synopsis/subjects
    csv_row  = CSV_META.get(title, {})
    synopsis = csv_row.get("synopsis", "")
    subjects = csv_row.get("subjects", "")
    authors  = csv_row.get("authors",  "")

    if synopsis or subjects:
        parts = [f"{title} by {authors}" if authors else title]
        if synopsis: parts.append(synopsis)
        if subjects: parts.append(f"Genres: {subjects}")
        vec = model.encode(
            [" | ".join(parts)],
            convert_to_numpy=True,
            normalize_embeddings=True
        )[0]
        profile_vecs.append(vec)
        profile_weights.append(w)
        rated_titles.append(title)
        rated_raw_ratings.append(rating)
        print(f"  [{rating} Stars] '{title}' by {authors}")
        continue

    # CSV empty, fall back to ISBNdb
    print(f"  [{rating} Stars] '{title}' not in dataset, no CSV metadata")
    info = search_isbndb_by_title(title)
    if info:
        parts = [f"{info['title']} by {info['authors']}" if info["authors"] else info["title"]]
        if info["synopsis"]: parts.append(info["synopsis"])
        if info["subjects"]: parts.append(f"Genres: {', '.join(info['subjects'][:8])}")
        vec = model.encode(
            [" | ".join(parts)],
            convert_to_numpy=True,
            normalize_embeddings=True
        )[0]
        profile_vecs.append(vec)
        profile_weights.append(w)
        rated_titles.append(info["title"])
        rated_raw_ratings.append(rating)
        print(f"         -> matched: {info['title']!r} by {info['authors']}")
        time.sleep(RATE_LIMIT)
    else:
        print(f"         -> not found anywhere, skipping")

print(f"\n{len(profile_vecs)} of {len(MY_RATINGS)} books contributed to profile")

  [5 Stars] Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback)) by J. K. Rowling
  [5 Stars] The Hobbit : The Enchanting Prelude to The Lord of the Rings by J.R.R. TOLKIEN
  [5 Stars] 'Divergent' not in dataset, no CSV metadata
         -> matched: 'Divergent' by Veronica Roth
  [4 Stars] Bridget Jones's Diary by Helen Fielding
  [3 Stars] The Da Vinci Code by Dan Brown
  [1 Stars] 'Twilight' not in dataset, no CSV metadata
         -> matched: 'Twilight' by Stephenie Meyer
  [2 Stars] To Kill a Mockingbird by Harper Lee
  [2 Stars] The Great Gatsby by F. Scott Fitzgerald
  [1 Stars] 'Pride and Prejudice' not in dataset, no CSV metadata
         -> matched: 'Pride and Prejudice (Penguin Classics)' by Jane Austen

9 of 9 books contributed to profile


Recommendation Task

In [127]:
# build weighted preference profile
w_arr = np.array(profile_weights)
profile = np.sum(np.vstack(profile_vecs) * w_arr[:, np.newaxis], axis=0)
profile /= np.linalg.norm(profile)

# mask out any dataset books already in preferences
candidate_mask = np.ones(len(books), dtype=bool)
for idx in dataset_rated_indices:
    candidate_mask[idx] = False

candidate_embeddings = embeddings[candidate_mask]
candidates = books[candidate_mask].copy().reset_index(drop=True)

# cosine similarity: profile vs every candidate
sims = cosine_similarity([profile], candidate_embeddings)[0]
candidates["score"] = sims

# predicted_rating = sum(sim × raw_rating) / sum(sim) for all positively similar books
_profile_mat = np.vstack(profile_vecs)
_raw_ratings_arr  = np.array(rated_raw_ratings)

def predict_rating(candidate_idx):
    emb = candidate_embeddings[candidate_idx]
    sims_to_rated = cosine_similarity([emb], _profile_mat)[0]
    pos = sims_to_rated > 0
    if not pos.any():
        return None
    w = sims_to_rated[pos]
    r = _raw_ratings_arr[pos]
    return round(float(np.dot(w, r) / w.sum()), 2)

# find which positively-rated books drove each recommendation
pos_vecs = np.vstack([v for v, w in zip(profile_vecs, profile_weights) if w > 1.0])
pos_titles = [t for t, w in zip(rated_titles, profile_weights) if w > 1.0]

def explain(candidate_idx):
    emb = candidate_embeddings[candidate_idx]
    sims_to_liked = cosine_similarity([emb], pos_vecs)[0]
    top2 = np.argsort(sims_to_liked)[::-1][:2]
    drivers = [pos_titles[i] for i in top2 if sims_to_liked[i] > 0.1]
    if drivers:
        short = [t if len(t) <= 35 else t[:32] + "…" for t in drivers]
        return "Because you liked: " + " & ".join(f'"{t}"' for t in short)
    return "Matches your overall profile"

# build output
top = candidates.sort_values("score", ascending=False).head(TOP_N).copy()
top["predicted_rating"] = [predict_rating(i) for i in top.index]
top["explanation"]      = [explain(i)         for i in top.index]
top.index = range(1, len(top) + 1)

out = top[["title", "author", "score", "predicted_rating", "explanation"]].copy()
out.columns = ["Title", "Author", "Similarity", "Predicted Rating", "Explaination"]

pd.set_option("display.max_colwidth", 55)
out

,Title,Author,Similarity,Predicted Rating,Explaination
1,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,0.602947,3.80,"Because you liked: ""Harry Potter and the Sorcerer's..."
2,"The Fellowship of the Ring (The Lord of the Rings, ...",J.R.R. TOLKIEN,0.544037,3.52,"Because you liked: ""The Hobbit : The Enchanting Pre..."
3,"The Two Towers (The Lord of the Rings, Part 2)",J.R.R. TOLKIEN,0.540859,3.51,"Because you liked: ""The Hobbit : The Enchanting Pre..."
4,"The Return of the King (The Lord of the Rings, Part 3)",J.R.R. TOLKIEN,0.506920,3.51,"Because you liked: ""The Hobbit : The Enchanting Pre..."
5,Dreamcatcher,Stephen King,0.503705,3.11,"Because you liked: ""Harry Potter and the Sorcerer's..."
6,The Phantom Tollbooth,Norton Juster,0.493979,3.48,"Because you liked: ""Divergent"" & ""Bridget Jones's D..."
7,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,0.493066,3.77,"Because you liked: ""Harry Potter and the Sorcerer's..."
8,The Eyre Affair: A Novel,Jasper Fforde,0.484080,2.98,"Because you liked: ""Harry Potter and the Sorcerer's..."
9,Harry Potter and the Order of the Phoenix (Book 5),J. K. Rowling,0.473456,3.44,"Because you liked: ""Harry Potter and the Sorcerer's..."
10,Dragonfly in Amber,DIANA GABALDON,0.469577,3.09,"Because you liked: ""The Hobbit : The Enchanting Pre..."


Rating Prediction Task

In [128]:
def predict_isbn(isbn):
    # fetch from ISBNdb
    print(f"querying ISBNdb for ISBN {isbn}")
    r = requests.get(
        f"https://api2.isbndb.com/book/{isbn}",
        headers={"Authorization": ISBNDB_API_KEY},
        timeout=10,
    )
    if r.status_code == 404:
        print("  not found on ISBNdb.")
        return None
    if r.status_code == 429:
        print("  rate limited — wait a moment and try again.")
        return None
    r.raise_for_status()

    book = r.json().get("book", {})
    title = (book.get("title")    or "").strip()
    authors = ", ".join(book.get("authors")  or [])
    synopsis = (book.get("synopsis") or "").strip()
    subjects = book.get("subjects")  or []

    if not title:
        print("  ISBNdb returned no title for this ISBN.")
        return None

    print(f"Title: {title}")
    print(f"Author: {authors or 'unknown'}")
    print(f"Synopsis: {'yes (' + str(len(synopsis)) + ' chars)' if synopsis else 'not available'}")
    print(f"Subjects: {', '.join(subjects[:5]) if subjects else 'not available'}")

    # build text and embed
    parts = [f"{title} by {authors}" if authors else title]
    if synopsis:
        parts.append(synopsis)
    if subjects:
        parts.append(f"Genres: {', '.join(subjects[:8])}")

    vec = model.encode(
        [" | ".join(parts)],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

    # cosine similarity against your taste profile
    similarity = float(cosine_similarity([vec], [profile])[0][0])

    # weighted KNN predicted rating
    sims_to_rated = cosine_similarity([vec], _profile_mat)[0]
    pos           = sims_to_rated > 0
    if pos.any():
        w = sims_to_rated[pos]
        r_vals = _raw_ratings_arr[pos]
        predicted = round(float(np.dot(w, r_vals) / w.sum()), 2)
    else:
        predicted = None

    # which liked books are most similar?
    sims_to_liked = cosine_similarity([vec], pos_vecs)[0]
    top2 = np.argsort(sims_to_liked)[::-1][:2]
    drivers = [pos_titles[i] for i in top2 if sims_to_liked[i] > 0.1]

    print(f"── Results ──")
    print(f"Similarity to profile: {similarity:.4f}")
    print(f"Predicted rating: {predicted} / 5" if predicted else "  Predicted rating : n/a")
    if drivers:
        print(f"Because you liked: {' & '.join(drivers)}")

    return {
        "title": title,
        "authors": authors,
        "similarity": similarity,
        "predicted": predicted,
        "drivers": drivers,
    }


In [129]:
predict_isbn("9781250301697")

querying ISBNdb for ISBN 9781250301697
Title: The Silent Patient
Author: Alex Michaelides
Synopsis: yes (1480 chars)
Subjects: Fiction, Thrillers, Psychological, Suspense
── Results ──
Similarity to profile: 0.2321
Predicted rating: 2.78 / 5
Because you liked: Bridget Jones's Diary & Divergent


{'title': 'The Silent Patient',
 'authors': 'Alex Michaelides',
 'similarity': 0.2321100224070963,
 'predicted': 2.78,
 'drivers': ["Bridget Jones's Diary", 'Divergent']}